# 01. Knowledge-based Agents dan Wumpus World


Notebook ini memperkenalkan apa itu agent berbasis pengetahuan dan dunia yang
dipakai sebagai contoh sepanjang materi, yaitu Wumpus World. Belum ada logika
formal di sini. Simbol seperti $P_{1,2}$ dan aturan $R_1$ sampai $R_5$ baru muncul
di notebook berikutnya.

**Batasan:** jangan pakai `Expr`, `expr()`, atau operator `|'==>'|`. Itu baru
diperkenalkan di Notebook 03. Contoh kode di sini pakai Python biasa saja.
`psource()` boleh dipakai untuk menampilkan source code.

**Output yang diharapkan:** pembaca paham kenapa agent butuh knowledge base, dan
hafal aturan main Wumpus World tanpa perlu buka slide lagi.

## Setup

Jalankan sel di bawah ini sekali di awal, sebelum sel mana pun yang lain.

Sel ini memasang dependensi yang diperlukan, mencari folder yang berisi
`logic.py` dan `utils.py`, lalu mengimpornya. Kalau notebook dibuka lewat Google
Colab, repo akan di-clone otomatis. Tidak ada yang perlu diubah di sini.

Environment sudah siap kalau baris terakhir output mencetak
`Check       : tt_entails(P & Q, Q) = True`.

In [ ]:
# =============================================================================
# Standard setup cell.
# Run this once, before any other cell in this notebook.
# =============================================================================
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kcv-if/Modul-Praktikum-KK-RKA-25.git"
ON_COLAB = "google.colab" in sys.modules


def ensure_dependencies():
    """Install only the packages this module actually uses."""
    required = {
        "networkx": "networkx",
        "numpy": "numpy",
        "pandas": "pandas",
        "matplotlib": "matplotlib",
        "ipywidgets": "ipywidgets",
        "PIL": "pillow",
        "pygments": "pygments",
    }
    missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", ", ".join(missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)


def find_environment(start):
    """Locate the folder that holds logic.py and utils.py, searching upward."""
    for root in [start, *start.parents]:
        for candidate in sorted(root.rglob("logic.py")):
            if (candidate.parent / "utils.py").exists():
                return candidate.parent
        if (root / ".git").exists():
            break
    return None


ensure_dependencies()

start_dir = Path.cwd()
if ON_COLAB:
    clone_dir = Path("Modul-Praktikum-KK-RKA-25")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)], check=True)
    start_dir = clone_dir

ENV_DIR = find_environment(start_dir)
if ENV_DIR is None:
    raise RuntimeError(
        "Environment folder not found. Make sure this notebook is opened from "
        "inside the Modul-Praktikum-KK-RKA-25 repository."
    )
if str(ENV_DIR) not in sys.path:
    sys.path.insert(0, str(ENV_DIR))

import itertools
import warnings

import pandas as pd

# qpsolvers is only used by the SVM code, which this module never touches.
warnings.filterwarnings("ignore", message="no QP solver found")

from logic import *
from notebook import psource
from utils import *

print("Environment :", ENV_DIR)
print("Python      :", sys.version.split()[0])
print("Check       : tt_entails(P & Q, Q) =", tt_entails(expr("P & Q"), expr("Q")))

---
# 1.1 Knowledge-based Agents

**Slide 3 sampai 5**

## Penjelasan

Yang perlu masuk:

- Definisi logical agent atau knowledge-based agent, dan tiga kemampuannya
  menurut slide 3: membuat representasi dari dunia yang kompleks, melakukan
  inferensi dari representasi menjadi representasi baru, dan menyimpulkan aksi
  berdasarkan representasi baru itu.
- Tiga hal yang bisa dilakukan KB agent menurut slide 4: menerima task baru dalam
  bentuk goal yang dideskripsikan eksplisit, mencapai kompetensi dengan cepat
  lewat diberi tahu atau belajar, dan beradaptasi terhadap perubahan environment
  dengan memperbarui pengetahuannya.
- Komponen utamanya, yaitu knowledge base, dan definisinya sebagai himpunan
  sentence.
- Knowledge representation language.
- Dua operasi dasar: TELL untuk menambah sentence, ASK untuk bertanya. Tekankan
  bahwa keduanya bisa melibatkan inference, yaitu menurunkan sentence baru dari
  sentence lama.

Kontras yang perlu dibuat: bedanya dengan agent yang cuma memetakan percept ke
aksi lewat tabel atau aturan if-then. Kenapa memisahkan pengetahuan dari program
itu menguntungkan.

## Contoh penerapan

Belum ada kode di sub-topik ini. Cukup satu ilustrasi non-formal, misalnya
menuliskan tiga sentence dalam bahasa Indonesia biasa lalu menunjukkan sentence
keempat yang bisa diturunkan darinya. Tujuannya membangun intuisi TELL dan ASK
sebelum ketemu sintaks formalnya.

---
# 1.2 Arsitektur KB Agent

**Slide 6, Figure 7.1**

## Penjelasan

Yang perlu masuk:

- Pseudocode `KB-AGENT(percept)` dari Figure 7.1, dibaca baris per baris.
- Tiga langkahnya: TELL percept ke KB, ASK aksi terbaik ke KB, TELL ke KB bahwa
  aksi itu sudah dilakukan.
- Fungsi `MAKE-PERCEPT-SENTENCE`, `MAKE-ACTION-QUERY`, `MAKE-ACTION-SENTENCE`.
- Peran counter waktu `t`. Jelaskan kenapa perlu, yaitu supaya KB bisa
  membedakan percept yang sama di waktu berbeda.

Pertanyaan yang bagus untuk dibahas di sini: kenapa aksi yang sudah dilakukan
perlu di-TELL balik ke KB, padahal agent sendiri yang melakukannya.

## Contoh penerapan

Tampilkan source code `KBAgentProgram` dari `logic.py` pakai `psource()`, lalu
cocokkan tiap barisnya dengan pseudocode Figure 7.1. Tidak perlu menjalankan
agent-nya, cukup dibaca.

---
# 1.3 Wumpus World

**Slide 7 dan 8, Figure 7.2**

## Penjelasan

Yang perlu masuk:

- Deskripsi dunia: gua berisi ruangan yang terhubung lorong.
- Wumpus, monster yang memakan siapa pun yang masuk ke ruangannya. Bisa
  ditembak, tapi agent cuma punya satu anak panah.
- Pit, lubang tanpa dasar yang menjebak siapa pun yang masuk.
- Emas, satu-satunya hal yang bikin dunia ini layak dimasuki.
- Layout Figure 7.2: grid 4 kali 4, agent mulai di [1,1] menghadap kanan.

Tampilkan gambar Figure 7.2 di sini, jangan cuma dideskripsikan. Bisa
di-screenshot dari slide lalu ditaruh di folder yang sama, atau digambar ulang
sebagai grid ASCII.

## Contoh penerapan

Buat representasi peta Figure 7.2 sebagai struktur data Python sederhana,
misalnya nested list atau dict yang menyimpan isi tiap kotak. Lalu tulis satu
fungsi kecil yang mencetak peta itu ke terminal dalam bentuk grid.

Ini bukan bagian dari materi logika, tapi berguna sebagai bahan visualisasi yang
bisa dipakai ulang di Notebook 04.

---
# 1.4 PEAS Definition

**Slide 9 sampai 12**

## Penjelasan

Empat slide ini isinya satu topik yang dipecah, jadi jelaskan sebagai satu
kesatuan. Jangan dipecah jadi empat sub-bagian di notebook.

Yang perlu masuk:

**Performance measure**
- Emas bernilai +1000
- Jatuh ke pit atau dimakan wumpus bernilai -1000
- Tiap aksi bernilai -1
- Memakai anak panah bernilai -10
- Permainan selesai kalau agent mati atau keluar dari gua

**Environment**
- Grid 4 kali 4
- Agent selalu mulai di [1,1] menghadap kanan
- Posisi emas dan wumpus dipilih acak dengan distribusi uniform, kecuali kotak
  awal
- Tiap kotak selain kotak awal bisa berisi pit dengan probabilitas 0.2

**Actuators**
- Forward, TurnLeft 90 derajat, TurnRight 90 derajat
- Agent mati kalau masuk kotak berisi pit atau wumpus hidup
- Kalau maju menabrak dinding, agent tidak berpindah
- Grab untuk mengambil emas di kotak yang sama
- Shoot untuk menembak lurus ke arah yang dihadapi
- Climb untuk keluar gua, hanya bisa dari [1,1]

**Sensors**
- Stench di kotak wumpus dan kotak yang bersebelahan langsung, bukan diagonal
- Breeze di kotak yang bersebelahan langsung dengan pit
- Glitter di kotak tempat emas berada
- Bump saat menabrak dinding
- Scream saat wumpus mati, terdengar di seluruh gua
- Format percept: `[Stench, Breeze, Glitter, Bump, Scream]`

Poin yang perlu ditekankan: kata "bersebelahan langsung, bukan diagonal" itu
menentukan aturan logika yang akan ditulis di Notebook 04. Kalau salah baca,
semua rumusnya ikut salah.

## Contoh penerapan

Tulis fungsi yang menerima peta dari sub-topik 1.3 dan sebuah posisi, lalu
mengembalikan percept di posisi itu dalam format list lima elemen. Uji dengan
beberapa posisi dari Figure 7.2, misalnya [1,1], [2,1], dan [1,2], lalu cocokkan
hasilnya dengan gambar di slide.

---
# 1.5 Karakteristik Environment dan Penjelajahan Agent

**Slide 13 sampai 16, Figure 7.3 dan 7.4**

## Penjelasan

Yang perlu masuk:

- Karakteristik Wumpus World: discrete, static, single-agent, sequential, dan
  partially observable. Jelaskan satu per satu, terutama dua yang terakhir.
- Sequential berarti reward bisa datang setelah banyak aksi.
- Partially observable berarti sebagian state tidak bisa dilihat langsung,
  contohnya posisi agent, kondisi kesehatan wumpus, dan ketersediaan anak panah.
  Ini alasan utama kenapa agent butuh KB.
- Figure 7.3: langkah pertama agent. Kondisi awal dengan percept
  `[None, None, None, None, None]`, lalu setelah satu langkah dengan percept
  `[None, Breeze, None, None, None]`.
- Figure 7.4: dua tahap berikutnya. Setelah langkah ketiga dengan percept
  `[Stench, None, None, None, None]`, dan setelah langkah kelima dengan percept
  `[Stench, Breeze, Glitter, None, None]`.
- Penutup dari slide 16: kalau informasi yang tersedia benar, kesimpulan yang
  ditarik agent dijamin benar. Ini sifat mendasar dari logical reasoning, dan
  yang membedakannya dari penalaran probabilistik.

Bagian paling berharga di sub-topik ini adalah trace-nya. Jangan cuma tempel
Figure 7.3 dan 7.4 lalu lanjut. Ceritakan langkah per langkah: percept apa yang
diterima, kesimpulan apa yang ditarik, kotak mana yang jadi OK, dan kenapa agent
memilih arah tertentu.

Satu momen yang perlu disorot: di Figure 7.4a agent bisa menyimpulkan wumpus ada
di [1,3] padahal dia belum pernah ke sana. Kesimpulan itu datang dari
menggabungkan dua percept di dua kotak berbeda, bukan dari satu percept saja.

## Contoh penerapan

Pakai fungsi percept dari sub-topik 1.4 untuk mensimulasikan urutan langkah agent
di Figure 7.3 dan 7.4. Cetak percept di tiap langkah, lalu di bawahnya tulis
kesimpulan yang ditarik agent dalam bahasa Indonesia biasa.

Belum perlu inference otomatis. Kesimpulannya boleh ditulis manual sebagai
string. Tujuannya menunjukkan alur berpikirnya, bukan mengotomasi.

---
# Latihan Soal

Isi bagian ini dengan minimal 3 soal. Susun dari yang paling ringan.

Format tiap soal: pernyataan soal, cell kosong untuk jawaban, lalu pembahasan
yang dibungkus `<details>`.

## Soal 1

Tingkat pemahaman. Usul arah soal: kenapa Wumpus World disebut partially
observable, dan sebutkan tiga aspek state yang tidak bisa diamati langsung oleh
agent.

## Soal 2

Tingkat penerapan. Usul arah soal: tulis definisi PEAS untuk satu domain lain,
misalnya robot vacuum cleaner atau agent yang bermain Minesweeper. Minta pembaca
menuliskan keempat komponennya lengkap seperti format slide 9 sampai 12.

Kalau mau bisa diverifikasi kode, minta juga implementasi fungsi percept untuk
domain itu.

## Soal 3

Tingkat analisis. Usul arah soal: agent berada di [2,1] dan merasakan breeze.
Kenapa agent tidak boleh langsung melangkah ke [2,2], padahal [2,2] belum tentu
berisi pit? Kaitkan jawabannya dengan performance measure di slide 9.

## Soal 4 (opsional)

Ruang untuk soal tambahan kalau dirasa perlu. Hapus kalau tidak dipakai.